#### Initialize the libraries, the environment and the client

In [1]:
import pandas as pd
import os
from pprint import pprint

try:
    from epo.tipdata.ops import OPSClient, models, exceptions
except ImportError:
    os.system('pip install epo-tipdata-ops')
    from epo.tipdata.ops import OPSClient, models, exceptions

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2
    
client = OPSClient(
    key=os.getenv("OPS_KEY"), secret=os.getenv("OPS_SECRET")
)

## 7. **Available Methods and Examples of Use**
The OPSClient class offers a rich set of methods for retrieving patent data, enabling you to fetch specific details or conduct comprehensive searches. Examples have to be run in TIP.


### 7.1 Published_data Method
#### 7.1.1 published_data(reference_type, input, endpoint, constituents, output_type)
* Retrieves published patent data (bibliographic data, abstracts, etc.) for a specific document.
* Parameters:<br>
```reference_type (str)```: Type of patent reference (e.g., "publication", "application", or "priority")<br>
```input (Epodoc or Docdb)```: Patent document identifier<br>
(détailler input pour montrer le no de brevet à entrer dans son format particulier pour que le service puisse identifier le document
expliquer le sens de Epodoc par rapport à DocDb por avoir des formats différents des brevets à chercher)
```endpoint (str, optional)```: Specifies the data to retrieve (e.g., "biblio" for bibliographic data, "abstract" for abstracts) - defaults to "biblio"<br>
```constituents (list[str], optional)```: List of additional elements to retrieve with bibliographic data (e.g., images)<br>
```output_type (str)```: Format of the returned data (raw XML or Pandas DataFrame)<br>
#### 7.1.2 Easy help by instruction
Typing a '?' sticking to a command, like this here client.published_data```?``` and running it in a cell triggers a supportive information detailing the parameters of the command in use:

In [2]:
client.published_data?

Signature:
client.published_data(
    reference_type: Literal['publication', 'application', 'priority'],
    input: epo_ops.models.Docdb | epo_ops.models.Epodoc,
    endpoint: str = 'biblio',
    constituents: list[str] | None = None,
    output_type: Literal['raw', 'dataframe'] = 'raw',
) -> str | pandas.core.frame.DataFrame
Docstring:
Retrieval service for published data.

Args:
    reference_type (str): Any of "publication", "application", or "priority".
    input (Epodoc or Docdb): The document number. Cannot be Original.
    endpoint (str, optional): "biblio", "abstract", "equivalents", "claims", "description",
                              "fulltext", "images". Defaults to "biblio".
    constituents (list[str], optional): List of "biblio", "abstract", "images", "full cycle".
    output_type (str, optional): "dataframe" or "raw" (xml). Defaults to "raw"

Returns:
    str | pd.DataFrame: an xml string or a Dataframe

Note:
1) only the endpoint "biblio" or "equivalents" use the cons

#### 7.1.3 Easy help as you type an instruction

Further, the above definition might be shown in a popup menu by putting the cursor on the opening parenthesis of the command "client.published_data()" and typing Shift + Tab. You may try it in the following cell:

In [3]:
# client.published_data()

#### 7.1.4 Example of retrieving bibliographic data with published_data method
This example retrieves bibliographic data (endpoint="biblio") for a published patent application identified by its document number stored in the Docdb object.

In [4]:
biblio = client.published_data(                           # Retrieve bibliography data
    reference_type="publication",                         # publication, application, priority
    input=models.Docdb("4106326", "EP", "A1", date=None), # original, docdb, epodoc
    endpoint="biblio",                                    # optional, defaults to biblio in case of published_data
    constituents=[],                                      # optional, list of constituents
    output_type="Dataframe",                              # optional, xml or DataFrame format
)
display(biblio)

,system,family-id,country,doc-number,kind,bibliographic-data|publication-reference,bibliographic-data|classifications-ipcr,bibliographic-data|patent-classifications,bibliographic-data|application-reference,bibliographic-data|priority-claims,bibliographic-data|parties,bibliographic-data|invention-title,bibliographic-data|references-cited,abstract|lang,abstract|p
0,ops.epo.org,82067485,EP,4106326,A1,{'document-id': [{'@document-id-type': 'docdb'...,"{'classification-ipcr': {'@sequence': '1', 'te...","{'patent-classification': [{'@sequence': '1', ...","{'@doc-id': '574299984', 'document-id': [{'@do...","{'priority-claim': [{'@sequence': '1', '@kind'...",{'applicants': {'applicant': [{'@sequence': '1...,"[{'@lang': 'de', '#text': 'AUTOMATISCHES FRAMI...","{'citation': [{'@cited-phase': 'undefined', '@...",en,A videoconferencing system includes different ...


In [5]:
# Print the columns of the DataFrame to verify presence
print("DataFrame Columns:", biblio.columns)

DataFrame Columns: Index(['system', 'family-id', 'country', 'doc-number', 'kind',
       'bibliographic-data|publication-reference',
       'bibliographic-data|classifications-ipcr',
       'bibliographic-data|patent-classifications',
       'bibliographic-data|application-reference',
       'bibliographic-data|priority-claims', 'bibliographic-data|parties',
       'bibliographic-data|invention-title',
       'bibliographic-data|references-cited', 'abstract|lang', 'abstract|p'],
      dtype='object')


The above code imports the Docdb class from the OPSClient class. Then, it defines the document number for the patent application we are interested in. It creates a Docdb object with that number. Finally, it calls the published_data method with the following arguments:<br>
* ```reference_type```: Set to "application" as we're looking for an application.<br>
* ```input```: The my_docdb object containing the document number.<br>
* ```endpoint```: Set to "biblio" to retrieve bibliographic data.<br>
* ```output_type```: Set to "dataframe" to get the data as a Pandas DataFrame for easier manipulation.<br>
The published_data method then calls the appropriate service and process the response. It will use the internal _get_content method to convert the response to the desired format (DataFrame in this case).<br>
The last line prints the first few rows of the biblio DataFrame, which contains information like system, family-id, country, doc-number, kind, publication-reference, classifications-ipcr, patent-classifications, application-reference, priority-claims, invention-title, etc. for the patent application.<br>


#### We want to concatenate input_country, input_doc_info and input_kind reads on three columns of the dataframe

In [6]:
# Extract the relevant fields from the DataFrame
input_country = biblio["country"].iloc[0]
input_doc_info = biblio["doc-number"].iloc[0]
input_kind = biblio["kind"].iloc[0]

# We want to concatenate input_country, input_doc_info and input_kind sread on three columns of the dataframe
print(f"input_country: {input_country}, input_doc_info {input_doc_info} and input_kind {input_kind }")

input_country: EP, input_doc_info 4106326 and input_kind A1


In [7]:
# Print the input Epodoc number
print(f"And obtain the input document: {input_country}{input_doc_info}{input_kind}")

And obtain the input document: EP4106326A1


#### We then loop through each column in the biblio DataFrame and print the column name and its first row value so as to expand the data

In [8]:
# Loop through each column in the biblio DataFrame and print the column name and its first row value
for column in biblio.columns:
    print(f"Column: {column}")
    
    # Get the first row value of the column
    value = biblio[column].iloc[0]
    
    # Print the value for inspection
    print();print(f"Value: {value}\n"[:1000])

Column: system

Value: ops.epo.org

Column: family-id

Value: 82067485

Column: country

Value: EP

Column: doc-number

Value: 4106326

Column: kind

Value: A1

Column: bibliographic-data|publication-reference

Value: {'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}

Column: bibliographic-data|classifications-ipcr

Value: {'classification-ipcr': {'@sequence': '1', 'text': 'H04N   7/    14            A I'}}

Column: bibliographic-data|patent-classifications

Value: {'patent-classification': [{'@sequence': '1', 'classification-scheme': {'@office': 'EP', '@scheme': 'CPCI'}, 'section': 'H', 'class': '04', 'subclass': 'N', 'main-group': '7', 'subgroup': '147', 'classification-value': 'I', 'generating-office': 'US'}, {'@sequence': '2', 'classification-scheme': {'@office': 'EP', '@scheme': 'CPCI'}, 'section': 'H', 'class': '04', 'subclass

The above loop iterates over ```biblio.columns```, which contains the list of column names in the DataFrame.

For each column, it prints the column name and then retrieves the value in the first row with ```.iloc[0]```, printing the content to inspect the structure.

This approach allows us to expend all columns and their corresponding collapsed values in the first row.

#### 7.1.5 Continuation of the prior example to retrieve classification data from the bibliographic data
This continued example shows a manageable number of rows and displays a limited number of classification sequences, in a concise output within an adjustable limit to show more or fewer classifications per row as needed.

#### 7.1.5.1 extract_classifications Function and  Limit Option
* extract_classifications Function: This function now directly returns a list of concatenated classification strings instead of printing detailed classification information.
* Limit Option: The function allows limiting the number of classifications (set to 5 as in your original code).

In [9]:
# Function to extract classification information and return unique concatenated classifications
def extract_classifications(classification_info, limit=None):
    classifications = set()  # Use a set to store unique classifications
    
    # Check if the classification info contains patent classifications
    if isinstance(classification_info, dict) and 'patent-classification' in classification_info:
        class_list = classification_info['patent-classification']
        
        # Limit the number of classifications if specified
        if limit:
            class_list = class_list[:limit]
        
        # Extract relevant classification details
        for classification in class_list:
            section = classification.get('section', '')
            _class = classification.get('class', '')
            subclass = classification.get('subclass', '')
            main_group = classification.get('main-group', '')
            subgroup = classification.get('subgroup', '')
            
            # Concatenate classification string
            classification_str = f"{section}{_class}{subclass}{main_group}/{subgroup}"
            classifications.add(classification_str)  # Use a set to ensure uniqueness
    
    return list(classifications)  # Convert back to a list to maintain original return format

#### 7.1.5.2 Concatenating Classifications
* Concatenating Classifications: After collecting all classifications, the results are stored in a DataFrame and output at the end.

In [10]:
# List to store concatenated classifications for each row
all_classifications = []

# Iterate through the rows of the DataFrame
for index, row in biblio.iterrows():
    if 'bibliographic-data|patent-classifications' in biblio.columns:
        classification_info = row['bibliographic-data|patent-classifications']
        # Extract unique classifications (limited to first 5)
        classifications = extract_classifications(classification_info, limit=5)
        all_classifications.extend(classifications)

# Convert the list of classifications to a DataFrame (if needed)
classification_df = pd.DataFrame({'Classification': all_classifications})

# Output the concatenated classifications
print(classification_df['Classification'])


0    H04N23/695
1      H04N7/15
2     H04N7/147
3    H04N13/204
Name: Classification, dtype: object


### 7.1.6 Second continuation of the prior example to retrieve references cited from the bibliographic data
This continued example shows a manageable number of rows and displays a limited number of citations, in a concise output.

#### 7.1.6.1 Normalize_doc_number function
* Function to normalize document numbers by stripping leading non-numeric characters for comparison

In [11]:
import pandas as pd

# Function to normalize document numbers by stripping leading non-numeric characters for comparison
def normalize_doc_number(doc_number):
    if doc_number:
        return ''.join(filter(str.isdigit, doc_number))  # Retain only the numeric part
    return doc_number

# Compile all references into a DataFrame
references_data = []

# Iterate through each row in the DataFrame
for index, row in biblio.iterrows():
    if 'bibliographic-data|references-cited' in biblio.columns:
        references_info = row['bibliographic-data|references-cited']

        if isinstance(references_info, dict) and 'citation' in references_info:
            citations = references_info['citation']

            if isinstance(citations, list):
                for ref in citations:
                    # For patent citations
                    if 'patcit' in ref:
                        patcit = ref['patcit']
                        if 'document-id' in patcit:
                            for doc in patcit['document-id']:
                                reference_entry = {
                                    'Document Number': doc.get('doc-number', None),
                                    'Normalized Document Number': normalize_doc_number(doc.get('doc-number', None)),
                                    'Country': doc.get('country', None),
                                    'Kind': doc.get('kind', None),
                                    'Date': doc.get('date', None),
                                    'Citation Category': ref.get('@cited-phase', None),
                                    'NPL Citation': None  # Set NPL Citation to None for patent citations
                                }
                                references_data.append(reference_entry)

                    # For non-patent literature citations (NPL)
                    elif 'nplcit' in ref:
                        nplcit = ref['nplcit']
                        reference_entry = {
                            'Document Number': None,
                            'Normalized Document Number': None,
                            'Country': None,
                            'Kind': None,
                            'Date': None,
                            'Citation Category': ref.get('@cited-phase', None),
                            'NPL Citation': nplcit.get('text', None)
                        }
                        references_data.append(reference_entry)

Handling Patent Citations ```patcit```:

* The patcit key contains a list of document-id, where each entry has doc-number, country, kind, and date. These values are extracted and added to the reference_entry.

Handling Non-Patent Literature Citations ```nplcit```:

* The nplcit key contains text for the citation, so when it's present, we create an entry for it in the DataFrame with fields such as NPL Citation.

#### 7.1.6.2 Handle references dataframe
* Convert list of dictionaries to a DataFrame
* Remove duplicates by 'Normalized Document Number', keeping the first occurrence
* Filter out rows where all fields are NaN or "None" for essential fields
* Drop the 'Normalized Document Number' column as it was only used for filtering duplicates
* Return the final DataFrame

In [12]:
# Convert list of dictionaries to a DataFrame
references_df = pd.DataFrame(references_data)

# Remove duplicates by 'Normalized Document Number', keeping the first occurrence
# This removes the duplicate rows like JPH08307902 and H08307902 by comparing only numeric parts of doc numbers
filtered_references_df = references_df.drop_duplicates(subset=['Normalized Document Number'])

# Filter out rows where all fields are NaN or "None" for essential fields
filtered_references_df = filtered_references_df.dropna(subset=['Document Number', 'Country', 'Kind', 'Date', 'NPL Citation'], how='all')

# Drop the 'Normalized Document Number' column as it was only used for filtering duplicates
filtered_references_df = filtered_references_df.drop(columns=['Normalized Document Number'])

# Return the final DataFrame
filtered_references_df

,Document Number,Country,Kind,Date,Citation Category,NPL Citation
0,US202762632025P,None,None,None,undefined,None
2,US8395653,None,None,20130312,undefined,None
4,US8842161,None,None,20140923,undefined,None
6,US9030520,None,None,20150512,undefined,None
8,US9542603,None,None,20170110,undefined,None
10,US9723260,None,None,20170801,undefined,None
12,US9800835,None,None,20171024,undefined,None
14,US10091412,None,None,20181002,undefined,None
16,US10122972,None,None,20181106,undefined,None
18,US10187579,None,None,20190122,undefined,None


### 7.1.7 Example of retrieving a set of claims with the published_data method
This example retrieves claims (endpoint="claims") for a published patent application identified by its document number stored in the Docdb object. One can run each cell separatelyand see the intermediate results, which helps understanding the flow of the program.

#### 7.1.7.1 Retrieves claims of EP4106326A1
* published_data is called for retrieving a set of claims from an EP publication number

In [13]:
biblio = client.published_data(                           # Retrieve bibliography data
    reference_type="publication",                         # publication, application, priority
    input=models.Docdb("4106326", "EP", "%%", date=None), # original, docdb, epodoc
    endpoint='claims',                                    # "biblio", "abstract", "equivalents", "claims", "description", "fulltext", "images",                                    # optional, defaults to biblio in case of published_data
    constituents=[],                                      # optional, list of constituents from ['biblio', 'abstract', 'images', 'full cycle'], 
    output_type="raw",                                    # optional, xml or DataFrame format
)
pprint(biblio)

('<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet type="text/xsl" '
 'href="../../../../../style/pub-ftxt-claims.xsl"?>\n'
 '<ops:world-patent-data xmlns="http://www.epo.org/exchange" '
 'xmlns:ops="http://ops.epo.org" '
 'xmlns:xlink="http://www.w3.org/1999/xlink"><ftxt:fulltext-documents '
 'xmlns="http://www.epo.org/fulltext" '
 'xmlns:ftxt="http://www.epo.org/fulltext"><ftxt:fulltext-document '
 'system="ops.epo.org" '
 'fulltext-format="text-only"><bibliographic-data><publication-reference '
 'data-format="docdb"><document-id><country>EP</country><doc-number>4106326</doc-number><kind>A1</kind></document-id></publication-reference></bibliographic-data><claims '
 'lang="EN"><claim><claim-text>1. A system (100) comprising:\n'
 'a first camera (118, 120) with a first field-of-view, FOV, (420, 422);\n'
 'a second camera (120, 118) with a second FOV (420, 422);\n'
 'a memory (210) storing computer-executable instructions (210); and\n'
 'a hardware processor (206) coupled to (200)

#### 7.1.7.2 Import and Parse claims XML
* Parse the XML response
* Define namespaces correctly
* Find all claim-text elements under the correct namespace

In [14]:
import xml.etree.ElementTree as ET

# Parse the XML response
root = ET.fromstring(biblio)

# Define namespaces correctly
namespaces = {
    'ftxt': 'http://www.epo.org/fulltext',
    'ops': 'http://ops.epo.org',
    'default': 'http://www.epo.org/exchange',
}

# Find all claim-text elements under the correct namespace
claims = root.findall(".//ftxt:claim-text", namespaces)

print(f"Number of claims found: {len(claims)}")

Number of claims found: 15


#### 7.1.7.3 Function extract_claim_number
* Extract the leading claimn number from the claim text (e.g., "1." from "1. Optische Platte...")
* Convert to an integer if possible

In [15]:
def extract_claim_number(claim_text):
    # Extract the leading number from the claim text (e.g., "1." from "1. Optische Platte...")
    claim_leading_number = claim_text.strip().split()[0].rstrip(".")
    
    # Convert to an integer if possible
    try:
        return int(claim_leading_number)
    except ValueError:
        return None

#### 7.1.7.4 Process and print the claims
* Check if claims were found
* Format each claim text separately
* Print the claim text

In [16]:
# Check if claims were found and print them
if not claims:
    print("No claims found.")
else:
    expected_claim_number = 1  # Start from the first claim
    for claim in claims:
        claim_text = "".join(claim.itertext())  # Extract the text content from the claim element
        
        claim_number = extract_claim_number(claim_text)

        # If the claim number doesn't match the expected number, stop
        if claim_number != expected_claim_number:
            print(f"Claim mismatch detected. Expected Claim {expected_claim_number} but got Claim {claim_number}.")
            break

        # Print the claim text
        print(f"Claim {expected_claim_number}: {claim_text}\n")

        # Increment the expected claim number
        expected_claim_number += 1


Claim 1: 1. A system (100) comprising:
a first camera (118, 120) with a first field-of-view, FOV, (420, 422);
a second camera (120, 118) with a second FOV (420, 422);
a memory (210) storing computer-executable instructions (210); and
a hardware processor (206) coupled to (200) the first camera (118, 120) and the second camera (120, 118), and the hardware processor (206) configured to execute the computer-executable instructions (210) to:
acquire at least a first frame (514) using the first camera (118, 120);
determine a first state associated with the at least a first frame (515);
acquire at least a second frame (516) using the second camera (120, 118);
determine a second state associated with the at least a second frame (518);
determine, based on the first state and the second state, change data indicative of a change in camera source from the first camera to the second camera (520);
determine output data (526) comprising a dissolve transition (526) from the at least a first frame (51

### 7.1.8 Example of fetching description data
This example retrieves description data (endpoint="description") for a published patent identified by its document number stored in the Docdb object.

#### 7.1.8.1 Fetch description data for a publication
* The function published_data is called for retrieving the description from an EP publication number

In [26]:
# Fetch description for a publication
description = client.published_data(
        reference_type='publication', # application, publication
        input=models.Docdb("4106326", "EP", "A1", date=None),
        endpoint='description',
        constituents=[],
        output_type="raw"
    )
description

'<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet type="text/xsl" href="../../../../../style/pub-ftxt-description.xsl"?>\n<ops:world-patent-data xmlns="http://www.epo.org/exchange" xmlns:ops="http://ops.epo.org" xmlns:xlink="http://www.w3.org/1999/xlink"><ftxt:fulltext-documents xmlns="http://www.epo.org/fulltext" xmlns:ftxt="http://www.epo.org/fulltext"><ftxt:fulltext-document system="ops.epo.org" fulltext-format="text-only"><bibliographic-data><publication-reference data-format="docdb"><document-id><country>EP</country><doc-number>4106326</doc-number><kind>A1</kind></document-id></publication-reference></bibliographic-data><description lang="EN"><p>PRIORITY</p><p>[0001]    This application claims priority to,  U.S. Provisional Patent Application No. 63/202,527, filed on June 15, 2021 , entitled &quot;Telephoto and Wide-Angle Automatic Framing&quot;, which is hereby incorporated by reference in its entirety. </p><p>BACKGROUND</p><p>[0002]    In a videoconference, framing refers 

#### 7.1.8.2 Import root
* Parse the XML response
* Define namespaces correctly

In [18]:
import xml.etree.ElementTree as ET
import re

# Parse the XML
root = ET.fromstring(description)

# Define namespaces correctly
namespaces = {
    'ftxt': 'http://www.epo.org/fulltext',
    'ops': 'http://ops.epo.org',
}

#### 7.1.8.3 Functions to extract text, expand_figure_range and extract_descriptions
* Extracts and concatenates all text from an XML element and its children
* Extract numeric parts from start and end figure numbers
* Generate a list of figure numbers in the range, preserving the prefix
* Iterate through all 'p' elements in the XML tree
* Check if the text starts with a section header
* Find all figure references in the text
* Add figure descriptions to the figures dictionary

In [19]:
def extract_text(element):
    # Extracts and concatenates all text from an XML element and its children
    # Removes leading and trailing whitespace    
    return ''.join(element.itertext()).strip()

def expand_figure_range(start, end):
    try:
        # Extract numeric parts from start and end figure numbers
        start_num = int(re.search(r'\d+', start).group())
        end_num = int(re.search(r'\d+', end).group())
        # Generate a list of figure numbers in the range, preserving the prefix
        return [f"{start[:-1]}{i}" for i in range(start_num, end_num + 1)]  # Keep the prefix of the figure number
    except Exception as e:
        # Handle any errors during range expansion
        print(f"Error expanding range: {e}. Start: {start}, End: {end}")
        return []  # Return an empty list if there's an error

def extract_descriptions(root):
    # Initialize dictionaries to store section and figure descriptions
    sections = {}
    figures = {}
    current_section = None
    description = []
    
    # Iterate through all 'p' elements in the XML tree
    for p in root.findall('.//ftxt:p', namespaces):
        # Extract text from the element and remove citation numbers
        text = extract_text(p)
        text = re.sub(r'\[\d{4}\]', '', text).strip()
        
        # Check if the text starts with a section header
        section_match = re.match(r'^(PRIORITY|BACKGROUND|BRIEF DESCRIPTION OF FIGURES|DETAILED DESCRIPTION)', text, re.I)
        
        # Find all figure references in the text
        figure_matches = re.findall(r'(Fig\.?|FIG\.?|Figure|Figures?)\s*([\dA-Z]+(?:[-–]\s*[\dA-Z]+)?(?:\s+to\s+[\dA-Z]+)?)', text, re.I)

        if section_match:
            # If a new section is found, save the previous section and start a new one
            if current_section:
                sections[current_section] = ' '.join(description).strip()
            current_section = section_match.group(1).strip()
            description = [text]
        elif figure_matches:
            # Process figure references. Collect figure descriptions only if the reference is valid
            for fig_prefix, fig_number in figure_matches:
                fig_number = fig_number.strip()
                # Ensure figure number is valid (only contains numbers and letters)
                if re.search(r'\d', fig_number):  # Ensure figure number contains a digit
                    if '-' in fig_number or '–' in fig_number:
                        # Handle figure ranges with hyphens
                        start, end = re.split(r'[-–]\s*', fig_number)
                        expanded_figures = expand_figure_range(start, end)
                        for fig in expanded_figures:
                            if fig and fig not in figures:  # Avoid duplicates
                                figures[fig] = text.strip()
                    elif 'to' in fig_number:
                        # Handle figure ranges with 'to'
                        start, end = fig_number.split('to')
                        expanded_figures = expand_figure_range(start.strip(), end.strip())
                    else:
                        # Single figure reference
                        expanded_figures = [fig_number]
                    
                    # Add figure descriptions to the figures dictionary
                    for fig in expanded_figures:
                        if fig and fig not in figures:  # Avoid duplicates
                            figures[fig] = text.strip()
                    else:
                        if fig_number not in figures:  # Avoid duplicates
                            figures[fig_number] = text.strip()
        elif current_section:
            # If not a section header or figure reference, add to current section description
            description.append(text)

    if current_section:
        sections[current_section] = ' '.join(description).strip()

    return sections, figures

#### 7.1.8.4 Process and print the description
* Extract descriptions and figure descriptions
* Display an overview of labeled sections
* Print first 200 characters for preview

In [20]:
# Extract descriptions and figure descriptions
sections, figures = extract_descriptions(root)

# Display an overview of labeled sections
print(f"Total sections found: {len(sections)}\n")
print("Overview of Labeled Sections:")

for section, content in sections.items():
    print(f"\n{section}:")
    print(f"   {content[:200]}...")  # Print first 200 characters for preview

Error expanding range: 'NoneType' object has no attribute 'group'. Start: 17, End: three
Total sections found: 4

Overview of Labeled Sections:

PRIORITY:
   PRIORITY This application claims priority to,  U.S. Provisional Patent Application No. 63/202,527, filed on June 15, 2021 , entitled "Telephoto and Wide-Angle Automatic Framing", which is hereby incor...

BACKGROUND:
   BACKGROUND In a videoconference, framing refers to the view or frame provided to the far end or site. Originally, framing was performed manually, first locally and then remotely. The pan, tilt, and zo...

BRIEF DESCRIPTION OF FIGURES:
   BRIEF DESCRIPTION OF FIGURES...

DETAILED DESCRIPTION:
   DETAILED DESCRIPTION A videoconferencing system may include multiple cameras with different fields-of-view (FOVs) of a conference room. These cameras may be operated to frame the conference room, a pe...


#### 7.1.8.5 Display an overview of figures

In [21]:
# Display an overview of figures
print(f"\nTotal figures found: {len(figures)}")
print("\nOverview of Figures:")

def sort_key(item):
    figure_num = re.findall(r'\d+', item[0])
    if figure_num:
        return (int(figure_num[0]), item[0])
    return (float('inf'), item[0])  # Place non-numeric figures at the end

for figure, description in sorted(figures.items(), key=sort_key):
    # Check if figure is a redundant reference
    if re.match(r'Figure\s+\d+', figure) and f"Figure {figure.split()[1]}:" in figures:
        continue  # Skip if it's redundant
    print(f"\nFigure {figure}:")
    print(f"   {description[:200]}...")  # Print first 200 characters for preview


Total figures found: 26

Overview of Figures:

Figure 1:
   The detailed description is set forth with reference to the accompanying figures. In the figures, the left-most digit(s) of a reference number identifies the figure in which the reference number first...

Figure 2:
   The detailed description is set forth with reference to the accompanying figures. In the figures, the left-most digit(s) of a reference number identifies the figure in which the reference number first...

Figure 3:
   The detailed description is set forth with reference to the accompanying figures. In the figures, the left-most digit(s) of a reference number identifies the figure in which the reference number first...

Figure 4:
   The detailed description is set forth with reference to the accompanying figures. In the figures, the left-most digit(s) of a reference number identifies the figure in which the reference number first...

Figure 5:
   The detailed description is set forth with reference to the accompa

### 7.1.9 Example of fetching abstract data
This example retrieves abstract data (endpoint="abstract") for a published patent identified by its document number stored in the Docdb object.

In [22]:
# Fetch description for a publication
abstract = client.published_data(
        reference_type='publication', # application, publication
        input=models.Docdb("4106326", "EP", "A1", date=None),
        endpoint='abstract',
        constituents=[],
        output_type="raw"
    )
pprint(abstract)

('<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet type="text/xsl" '
 'href="../../../../../style/exchange.xsl"?>\n'
 '<ops:world-patent-data xmlns="http://www.epo.org/exchange" '
 'xmlns:ops="http://ops.epo.org" xmlns:xlink="http://www.w3.org/1999/xlink">\n'
 '    <exchange-documents>\n'
 '        <exchange-document country="EP" doc-number="4106326" kind="A1">\n'
 '            <bibliographic-data>\n'
 '                <publication-reference>\n'
 '                    <document-id document-id-type="docdb">\n'
 '                        <country>EP</country>\n'
 '                        <doc-number>4106326</doc-number>\n'
 '                        <kind>A1</kind>\n'
 '                        <date>20221221</date>\n'
 '                    </document-id>\n'
 '                    <document-id document-id-type="epodoc">\n'
 '                        <doc-number>EP4106326</doc-number>\n'
 '                        <date>20221221</date>\n'
 '                    </document-id>\n'
 '         

In [23]:
import xml.etree.ElementTree as ET
import re

# Define namespaces correctly
namespace = {'ops': 'http://ops.epo.org', 'ex': 'http://www.epo.org/exchange'}

# Parse the XML
root = ET.fromstring(abstract)

# Extract bibliographic data
doc_info = root.find('.//ex:document-id[@document-id-type="docdb"]', namespace)
country = doc_info.find('ex:country', namespace).text
doc_number = doc_info.find('ex:doc-number', namespace).text
kind = doc_info.find('ex:kind', namespace).text
date = doc_info.find('ex:date', namespace).text

# Extract abstract and language
abstract_elem = root.find('.//ex:abstract', namespace)
abstract_lang = abstract_elem.attrib['lang']
abstract_text = abstract_elem.find('ex:p', namespace).text

# Display the extracted information
print(f"Country: {country}")
print(f"Document Number: {doc_number}")
print(f"Kind: {kind}")
print(f"Publication Date: {date}")
print(f"Abstract Language: {abstract_lang}")
print(f"Abstract Text: {abstract_text}")


Country: EP
Document Number: 4106326
Kind: A1
Publication Date: 20221221
Abstract Language: en
Abstract Text: A videoconferencing system includes different cameras with different fields-of-view (FOVs). Frames of images acquired by these cameras may be processed to determine states, such as number of users present, talking, and so forth. Based on these states, frames from one camera may be selected to send to a far site. If the two states indicate no change in camera source from a first time to a second time, the current frame from the first camera that is sent to the far site is followed by another frame from the same camera. If the two states indicate a change in camera source from the first time to the second time, the current frame from the first camera that is sent to the far site is followed by a frame from the different camera. Depending on the two states and the cameras used, different transition effects may be applied to improve presentation.


In [24]:
# Fetch biblio and abstract for a publication
abstract = client.published_data(
        reference_type='publication', # application, publication
        input=models.Docdb("4106326", "EP", "A1", date=None),
        endpoint="biblio",
        constituents=["abstract"],
        output_type="dataframe"
        )
display(abstract)

# Check if the abstract DataFrame is not empty before proceeding
if not abstract.empty:
    # Extract the application number from the 'bibliographic-data|application-reference' column
    app_ref = abstract.iloc[0]['bibliographic-data|application-reference']
    
    appln_auth = None
    for doc_id in app_ref['document-id']:
        if doc_id['@document-id-type'] == 'docdb':
            appln_auth = doc_id['country']
            break  # Found the appln_auth, no need to continue
    
    # Extract appln_nr from the document-id list
    appln_number = None
    for doc_id in app_ref['document-id']:
        if doc_id['@document-id-type'] == 'docdb':
            appln_number = doc_id['doc-number']
            break  # Found the appln_number, no need to continue
            
    appln_kind = None
    for doc_id in app_ref['document-id']:
        if doc_id['@document-id-type'] == 'docdb':
            appln_kind = doc_id['kind']
            break  # Found the appln_kind, no need to continue
            
    # Extract publication date from 'bibliographic-data|publication-reference'
    pub_ref = abstract.iloc[0]['bibliographic-data|publication-reference']
    publication_date = None
    for doc_id in pub_ref['document-id']:
        if doc_id['@document-id-type'] == 'docdb':
            publication_date = doc_id['date']  # Extracting the publication date
            break  # Found the publication date, no need to continue

    # Example structure for TLS203-like format
    tls203_format = pd.DataFrame({      
        'docdb_appln_nr': [f"{appln_auth}.{appln_number}.{appln_kind}"],  # Application number (publication number here)
        'epodoc_appln_nr': [f"{appln_auth}{appln_number}{appln_kind}"],  # Application number (publication number here)          
        'abstract_text': [abstract.iloc[0]['abstract|p']],  # Extracting abstract text from 'abstract' DataFrame
        'abstract_lg': [abstract.iloc[0]['abstract|lang'].upper()]  # Extracting abstract language from 'abstract' DataFrame
    })

    # Adjusting pandas settings to show full text without truncation
    pd.set_option('display.max_colwidth', None)

    # Displaying the TLS203-like DataFrame
    print("tls203_format:", tls203_format)

    display(tls203_format)

,system,family-id,country,doc-number,kind,bibliographic-data|publication-reference,bibliographic-data|classifications-ipcr,bibliographic-data|patent-classifications,bibliographic-data|application-reference,bibliographic-data|priority-claims,bibliographic-data|parties,bibliographic-data|invention-title,bibliographic-data|references-cited,abstract|lang,abstract|p
0,ops.epo.org,82067485,EP,4106326,A1,{'document-id': [{'@document-id-type': 'docdb'...,"{'classification-ipcr': {'@sequence': '1', 'te...","{'patent-classification': [{'@sequence': '1', ...","{'@doc-id': '574299984', 'document-id': [{'@do...","{'priority-claim': [{'@sequence': '1', '@kind'...",{'applicants': {'applicant': [{'@sequence': '1...,"[{'@lang': 'de', '#text': 'AUTOMATISCHES FRAMI...","{'citation': [{'@cited-phase': 'undefined', '@...",en,A videoconferencing system includes different ...


tls203_format:   docdb_appln_nr epodoc_appln_nr  \
0  EP.22179301.A     EP22179301A   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            abstract_text  \
0  A videoconferencing system includes different ca

,docdb_appln_nr,epodoc_appln_nr,abstract_text,abstract_lg
0,EP.22179301.A,EP22179301A,"A videoconferencing system includes different cameras with different fields-of-view (FOVs). Frames of images acquired by these cameras may be processed to determine states, such as number of users present, talking, and so forth. Based on these states, frames from one camera may be selected to send to a far site. If the two states indicate no change in camera source from a first time to a second time, the current frame from the first camera that is sent to the far site is followed by another frame from the same camera. If the two states indicate a change in camera source from the first time to the second time, the current frame from the first camera that is sent to the far site is followed by a frame from the different camera. Depending on the two states and the cameras used, different transition effects may be applied to improve presentation.",EN
